# Entrenamiento de Redes Neuronales de Grafos


Este estregable tiene por objetivo entrenar redes neuronales de grafos GNN (Graph Neuronal Networks) para predecir transacciones fraudulentas de Bitcoin en el grafo de Elliptic.

## Grafo de Elliptic


Primero antes que nada, se debe construir el grafo a partir del dataset y transformarlo para usarlo de input en lo modelos.

La idea es obtener un único objeto del tipo 'Data' de la librería PyTorch Geomettric (PyG) que contenga toda la info necesaria para describir el grafo.

Este tipo objeto de datos contiene cuatro atributos: 
1) edge_index contiene información sobre la conectividad del grafo; es decir, una tupla de índices de nodo de origen y destino para cada arista.
2) características de los nodos como 'x' (cada uno de los nodos tiene asignado un vector de características de n dimensiones). 
3) las etiquetas de los nodos como 'y' (cada nodo tiene asignado exactamente una clase).
4) También existe un atributo adicional llamado 'train_mask', que describe para qué nodos ya conocemos sus asignaciones de comunidad.

Se procede a construir el 'Data'.


In [1]:
# Importar librerías
import pandas as pd
import torch

In [2]:
# Rutas a csv (setear según corresponda)
FEATURES_CSV = r"D:\Documents\Cursos\Diplomatura FAMAF\Mentoria\elliptic-gnn-fraud-detector\data\elliptic\elliptic_txs_features.csv"
EDGELIST_CSV = r"D:\Documents\Cursos\Diplomatura FAMAF\Mentoria\elliptic-gnn-fraud-detector\data\elliptic\elliptic_txs_edgelist.csv"
CLASSES_CSV = r"D:\Documents\Cursos\Diplomatura FAMAF\Mentoria\elliptic-gnn-fraud-detector\data\elliptic\elliptic_txs_classes.csv"

In [3]:
# Cargar csv's
df_features = pd.read_csv(FEATURES_CSV, header=None)
df_edgelist = pd.read_csv(EDGELIST_CSV, header=None)
df_classes = pd.read_csv(CLASSES_CSV, header=None)

### Cargar features y construir 'x'

In [4]:
# Cargar csv 
df_features = pd.read_csv(FEATURES_CSV, header=None)

df_features.head()

,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117


In [5]:
# Asignar nombres de columnas
df_features.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(165)]

# Guardar mapeo txId → índice
txid_list = df_features['txId'].tolist()
txid_to_idx = {txid: idx for idx, txid in enumerate(txid_list)}

# Extraer solo las columnas de features
x = torch.tensor(df_features.iloc[:, 2:].values, dtype=torch.float)

print(f"x shape: {x.shape}")

x shape: torch.Size([203769, 165])


Con lo anterior se obtiene:
* x: tensor (N, 166)
* txid_to_idx: diccionario para mapear desde cualquier txId a índice de fila.
* df_feature: útil para usar time_step después.

### Construir el tensor 'y'

In [6]:
# Cargar CSV con encabezado
df_classes = pd.read_csv(CLASSES_CSV)

df_classes.head()

,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown


In [7]:
# Mapear clase textual a entero: '1' → 1 (fraude), '2' → 0 (lícito), 'unknown' → -1
label_map = {'1': 1, '2': 0, 'unknown': -1}
df_classes['label'] = df_classes['class'].map(label_map)

# Alinear con df_features según txId
labels_aligned = df_features['txId'].map(dict(zip(df_classes['txId'], df_classes['label'])))

# Convertir a tensor
y = torch.tensor(labels_aligned.fillna(-1).astype(int).values)

# Verificar resultado
print("y shape:", y.shape)
print("Clases:", torch.unique(y, return_counts=True))

y shape: torch.Size([203769])
Clases: (tensor([-1,  0,  1]), tensor([157205,  42019,   4545]))


Se obtiene:
* y: tensor (N,) con valores en {0, 1, -1}
* Valores -1 corresponden a nodos sin etiqueta (desconocidos).

### Cargar aristas y construir 'edge_index'

In [8]:
# Cargar CSV con encabezado
df_edges = pd.read_csv(EDGELIST_CSV)

df_edges.head()

,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


In [9]:
# Mapear txId1 y txId2 a índices usando el diccionario creado antes
src = df_edges['txId1'].map(txid_to_idx)
dst = df_edges['txId2'].map(txid_to_idx)

# Filtrar pares válidos (aquellos que estén en features.csv). El filtro garantiza que todos los nodos en las aristas existan en el grafo.
valid = src.notna() & dst.notna()
src = src[valid].astype(int)
dst = dst[valid].astype(int)

# Crear tensor edge_index de tamaño (2, num_edges)
edge_index = torch.tensor([src.values, dst.values], dtype=torch.long)


print("edge_index shape:", edge_index.shape)

edge_index shape: torch.Size([2, 234355])


C:\Users\Nico\AppData\Local\Temp\ipykernel_5632\3060246405.py:11: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  edge_index = torch.tensor([src.values, dst.values], dtype=torch.long)


### Construcción de 'train_mask', 'val_mask', 'test_mask' en función de 'time_step'

Se usará el criterio estándar propuesto en papers y foros para Elliptic:
* train: time_step 1–35
* val: time_step 36–40
* test: time_step 41–49
* Solo se incluyen nodos etiquetados (y != -1)

In [12]:
# Obtener time_step como tensor
time_steps = torch.tensor(df_features['time_step'].values)

# Tensor booleano
y_mask = y != -1  # Tensor booleano

# Crear máscaras booleanas con lógica temporal + etiqueta conocida
train_mask = ((time_steps >= 1) & (time_steps <= 35)) & y_mask # nodos donde se entrena
val_mask   = ((time_steps >= 36) & (time_steps <= 40)) & y_mask # nodos donde se evalúa durante el entrenamiento (early stopping, monitorear overfitting)
test_mask  = ((time_steps >= 41) & (time_steps <= 49)) & y_mask # nodos donde se reporta la performance final

# Verificar
print("Train:", train_mask.sum().item(), "nodos")
print("Val:", val_mask.sum().item(), "nodos")
print("Test:", test_mask.sum().item(), "nodos")


Train: 31235 nodos
Val: 5356 nodos
Test: 9973 nodos


La distribución es razonable y se trabaja con un total de 46564 nodos con etiqueta

### Construcción del 'Data'

In [13]:
from torch_geometric.data import Data

data = Data(
    x=x,
    edge_index=edge_index,
    y=y,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)

print(data)


Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], train_mask=[203769], val_mask=[203769], test_mask=[203769])


Ya está todo listo para empezar a entrenar modelos GNN.

## Graph Convolutional Network - GCNConv
